<cell_type>markdown</cell_type># Step 6: 모델 배포 & Gradio 데모

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- **SageMaker Endpoint** 배포 과정
- **실시간 추론** 아키텍처
- **Gradio**를 활용한 데모 UI 구축

## 배포 아키텍처

```
┌─────────────────────────────────────────────────────────────┐
│                    SageMaker Endpoint                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [사용자]                                                    │
│     │                                                       │
│     │ 이미지 업로드                                          │
│     ▼                                                       │
│  ┌─────────────┐      ┌─────────────┐      ┌─────────────┐  │
│  │   Gradio    │ ──▶  │  SageMaker  │ ──▶  │   Model     │  │
│  │   Web UI    │      │  Endpoint   │      │  (PyTorch)  │  │
│  └─────────────┘      └─────────────┘      └─────────────┘  │
│         │                    │                    │         │
│         │◀───────────────────┼────────────────────┘         │
│         │               결과 반환                            │
│         ▼                                                   │
│  ┌─────────────┐                                            │
│  │ REAL / FAKE │                                            │
│  │  + 확신도    │                                            │
│  └─────────────┘                                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## Endpoint 비용

| 인스턴스 | 시간당 비용 | 비고 |
|---------|-----------|------|
| ml.g4dn.xlarge | ~$0.74 | GPU (추론용) |

> ⚠️ **중요**: 실습 후 반드시 Endpoint를 삭제하세요!

In [ ]:
import json
import os
import sagemaker
from sagemaker.pytorch import PyTorchModel
from datetime import datetime
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '6_demo'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

sagemaker_session = sagemaker.Session()
role = config['role']

# 최고 성능 기법의 모델 사용
best_method = config.get('best_method', 'full')
training_results = config.get('training_results', {})

if training_results and best_method in training_results:
    model_data = training_results[best_method]['model_data']
else:
    model_data = config['model_data']

# 고유한 Endpoint 이름 생성 (충돌 방지)
ENDPOINT_NAME = f"deepfake-{best_method}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"배포할 모델: {best_method.upper()} Fine-tuned")
print(f"모델 경로: {model_data}")
print(f"Endpoint Name: {ENDPOINT_NAME}")

<cell_type>markdown</cell_type>## 6.1 SageMaker Endpoint 배포

### 배포 과정 이해

1. **PyTorchModel 생성**: S3의 model.tar.gz와 inference.py 지정
2. **deploy() 호출**: SageMaker가 자동으로:
   - EC2 인스턴스 프로비저닝 (ml.g4dn.xlarge)
   - Docker 컨테이너 시작
   - 모델 로드 및 웜업
3. **Endpoint 생성**: HTTPS 엔드포인트 URL 제공

### inference.py 역할

```python
def model_fn(model_dir):     # 모델 로드
def input_fn(data, type):    # 입력 전처리
def predict_fn(data, model): # 추론 실행
def output_fn(pred, type):   # 출력 후처리
```

> ⏱️ 배포에 약 5-10분 소요됩니다.

In [ ]:
# ============================================
# 🚀 SageMaker Endpoint 배포
# ============================================

pytorch_model = PyTorchModel(
    model_data=model_data,  # 최고 성능 기법의 모델
    role=role,
    entry_point='inference.py',
    source_dir='.',
    framework_version='2.0.0',
    py_version='py310'
)

print(f"배포할 모델: {best_method.upper()} Fine-tuned")
print("Endpoint 배포 중... (약 5-10분 소요)")

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
    endpoint_name=ENDPOINT_NAME
)

print(f"\n✅ Endpoint 배포 완료!")
print(f"   Endpoint 이름: {predictor.endpoint_name}")
print(f"   인스턴스: ml.g4dn.xlarge (NVIDIA T4 GPU)")

# config에 endpoint 이름 저장
config['endpoint_name'] = ENDPOINT_NAME
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"\n💾 Endpoint 이름이 config.json에 저장되었습니다.")

<cell_type>markdown</cell_type>## 6.2 Gradio 데모 실행

### Gradio란?
- ML 모델을 위한 **웹 UI 프레임워크**
- Python 코드 몇 줄로 데모 인터페이스 생성
- `share=True`로 외부 공유 가능한 URL 생성

### 데모 흐름

```
사용자가 이미지 업로드
      ↓
이미지 → JPEG 바이트로 변환
      ↓
SageMaker Endpoint 호출
      ↓
결과 (REAL/FAKE + 확신도) 표시
```

> 💡 `share=True`를 사용하면 임시 공개 URL이 생성되어 다른 사람과 공유할 수 있습니다.

In [ ]:
# Gradio 데모 실행 (노트북 내에서)
import gradio as gr
import boto3
import base64
from io import BytesIO
from PIL import Image

runtime = boto3.client('sagemaker-runtime')

def detect_deepfake(image):
    """딥페이크 탐지 함수"""
    if image is None:
        return "이미지를 업로드해주세요."
    
    # 이미지를 base64로 인코딩
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    img_bytes = buffered.getvalue()
    
    # Endpoint 호출
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/x-image',
        Body=img_bytes
    )
    
    result = json.loads(response['Body'].read().decode())
    
    # 결과 포맷팅
    prediction = result.get('prediction', 'Unknown')
    confidence = result.get('confidence', 0)
    
    if prediction == 'FAKE':
        return f"🚨 FAKE 탐지!\n확신도: {confidence:.1%}"
    else:
        return f"✅ REAL\n확신도: {confidence:.1%}"

# Gradio 인터페이스
demo = gr.Interface(
    fn=detect_deepfake,
    inputs=gr.Image(type="pil", label="이미지 업로드"),
    outputs=gr.Textbox(label="탐지 결과"),
    title="🎭 딥페이크 탐지 데모",
    description="이미지를 업로드하면 딥페이크 여부를 판별합니다.\n(KoDF Fine-tuned 모델 사용)"
)

# 노트북에서 실행
demo.launch(share=True)

<cell_type>markdown</cell_type>## 6.3 리소스 정리 (중요!)

### 비용 절감을 위해 반드시 실행하세요

Endpoint는 **실행 중인 동안 계속 과금**됩니다.
실습이 끝나면 아래 셀의 주석을 해제하고 실행하여 Endpoint를 삭제하세요.

| 리소스 | 시간당 비용 | 삭제 후 |
|--------|-----------|--------|
| ml.g4dn.xlarge Endpoint | ~$0.74 | $0 |

> ⚠️ **Workshop 종료 시 반드시 Endpoint를 삭제**하세요!

In [ ]:
# ⚠️ 실습 완료 후 반드시 실행하세요! (비용 절감)
# 아래 주석을 해제하고 실행하면 Endpoint가 삭제됩니다.

# predictor.delete_endpoint()
# print(f"✅ Endpoint '{ENDPOINT_NAME}' 삭제 완료!")
# print("더 이상 비용이 발생하지 않습니다.")

<cell_type>markdown</cell_type>## 🎉 Workshop 완료!

축하합니다! 딥페이크 탐지 모델 Fine-tuning Workshop을 완료했습니다.

### 학습한 내용 요약

| 단계 | 내용 |
|------|------|
| **1. 데이터 준비** | S3에서 데이터 다운로드, 구조 이해 |
| **2. Before 평가** | Pretrained 모델의 한계 확인 (~70%) |
| **3. Fine-tuning** | Full, Freeze, LoRA 세 가지 기법 비교 |
| **4. After 평가** | Fine-tuned 모델 성능 확인 (~90%) |
| **5. 결과 비교** | 기법별 장단점 분석 |
| **6. 데모 배포** | SageMaker Endpoint + Gradio UI |

### 핵심 학습 포인트

1. **Domain Shift 문제**: Pretrained 모델은 다른 도메인에서 성능 저하
2. **Fine-tuning 효과**: 타겟 도메인 데이터로 학습 시 성능 크게 향상
3. **기법 선택**: 상황에 따라 Full, Freeze, LoRA 중 선택
4. **SageMaker 활용**: Experiments, Model Registry, Spot Instance

### 다음 단계 (심화)

- 더 많은 한국인 데이터로 학습
- 다른 모델 아키텍처 실험 (ViT, ConvNeXt)
- A/B 테스트 및 프로덕션 배포

감사합니다! 🙏